1. Из ноутбуков по практике "Рекуррентные и одномерные сверточные нейронные сети" выберите лучшую сеть, либо создайте свою.
2. Запустите раздел "Подготовка"
3. Подготовьте датасет с параметрами `VOCAB_SIZE=20'000`, `WIN_SIZE=1000`, `WIN_HOP=100`, как в ноутбуке занятия, и обучите выбранную сеть. Параметры обучения можно взять из практического занятия. Для  всех обучаемых сетей в данной работе они должны быть одни и теже.
4. Поменяйте размер словаря tokenaizera (`VOCAB_SIZE`) на `5000`, `10000`, `40000`.  Пересоздайте датасеты, при этом оставьте `WIN_SIZE=1000`, `WIN_HOP=100`.
Обучите выбранную нейронку на этих датасетах.  Сделайте выводы об  изменении  точности распознавания авторов текстов. Результаты сведите в таблицу
5. Поменяйте длину отрезка текста и шаг окна разбиения текста на векторы  (`WIN_SIZE`, `WIN_HOP`) используя значения (`500`,`50`) и (`2000`,`200`). Пересоздайте датасеты, при этом оставьте `VOCAB_SIZE=20000`. Обучите выбранную нейронку на этих датасетах. Сделайте выводы об  изменении точности распознавания авторов текстов.

Результаты всей работы сведите в таблицу.

## 1. Импорт библиотек

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras import utils
import gdown
import os
import re
import time
import matplotlib.pyplot as plt
from IPython.display import display

%matplotlib inline
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## 2. Загрузка датасета

In [12]:
gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l7/writers.zip', None, quiet=True)

'writers.zip'

## 3. Распаковка

In [13]:
!unzip -o writers.zip -d writers/

Archive:  writers.zip
  inflating: writers/(Клиффорд_Саймак) Обучающая_5 вместе.txt  
  inflating: writers/(Клиффорд_Саймак) Тестовая_2 вместе.txt  
  inflating: writers/(Макс Фрай) Обучающая_5 вместе.txt  
  inflating: writers/(Макс Фрай) Тестовая_2 вместе.txt  
  inflating: writers/(О. Генри) Обучающая_50 вместе.txt  
  inflating: writers/(О. Генри) Тестовая_20 вместе.txt  
  inflating: writers/(Рэй Брэдберри) Обучающая_22 вместе.txt  
  inflating: writers/(Рэй Брэдберри) Тестовая_8 вместе.txt  
  inflating: writers/(Стругацкие) Обучающая_5 вместе.txt  
  inflating: writers/(Стругацкие) Тестовая_2 вместе.txt  
  inflating: writers/(Булгаков) Обучающая_5 вместе.txt  
  inflating: writers/(Булгаков) Тестовая_2 вместе.txt  


In [14]:
FILE_DIR  = 'writers'
SIG_TRAIN = 'обучающая'
SIG_TEST  = 'тестовая'

## 4. Загрузка текстов

In [19]:
CLASS_LIST = []
text_train = []
text_test = []

file_list = os.listdir(FILE_DIR)
print("Найденные файлы:", file_list)

for file_name in file_list:
    if not file_name.endswith('.txt'):
        continue

    import re
    match = re.match(r'\((.*?)\)\s+(\S+)_\d+', file_name)

    if match:
        class_name = match.group(1).strip()
        subset_name = match.group(2).lower()

        is_train = 'обучающая' in subset_name
        is_test = 'тестовая' in subset_name

        if is_train or is_test:
            if class_name not in CLASS_LIST:
                print(f'Добавление класса "{class_name}"')
                CLASS_LIST.append(class_name)
                text_train.append('')
                text_test.append('')

            cls = CLASS_LIST.index(class_name)
            print(f'Добавление файла "{file_name}" в класс "{CLASS_LIST[cls]}", {"обучающая" if is_train else "тестовая"} выборка.')

            with open(f'{FILE_DIR}/{file_name}', 'r', encoding='utf-8') as f:
                text = f.read()

            subset = text_train if is_train else text_test
            subset[cls] += ' ' + text.replace('\n', ' ')
    else:
        print(f'Не удалось распарсить файл: {file_name}')

Найденные файлы: ['(Макс Фрай) Обучающая_5 вместе.txt', '(Клиффорд_Саймак) Обучающая_5 вместе.txt', '(О. Генри) Обучающая_50 вместе.txt', '(Рэй Брэдберри) Обучающая_22 вместе.txt', '(Стругацкие) Обучающая_5 вместе.txt', '(Булгаков) Тестовая_2 вместе.txt', '(Макс Фрай) Тестовая_2 вместе.txt', '(О. Генри) Тестовая_20 вместе.txt', '(Клиффорд_Саймак) Тестовая_2 вместе.txt', '(Булгаков) Обучающая_5 вместе.txt', '(Стругацкие) Тестовая_2 вместе.txt', '(Рэй Брэдберри) Тестовая_8 вместе.txt']
Добавление класса "Макс Фрай"
Добавление файла "(Макс Фрай) Обучающая_5 вместе.txt" в класс "Макс Фрай", обучающая выборка.
Добавление класса "Клиффорд_Саймак"
Добавление файла "(Клиффорд_Саймак) Обучающая_5 вместе.txt" в класс "Клиффорд_Саймак", обучающая выборка.
Добавление класса "О. Генри"
Добавление файла "(О. Генри) Обучающая_50 вместе.txt" в класс "О. Генри", обучающая выборка.
Добавление класса "Рэй Брэдберри"
Добавление файла "(Рэй Брэдберри) Обучающая_22 вместе.txt" в класс "Рэй Брэдберри", обуча

In [23]:
CLASS_COUNT = len(CLASS_LIST)

## 5. Вывод классов

In [21]:
print(CLASS_LIST)

['Макс Фрай', 'Клиффорд_Саймак', 'О. Генри', 'Рэй Брэдберри', 'Стругацкие', 'Булгаков']


## 6. Измеритель времени

In [25]:
class timex:
    def __enter__(self):
        self.t = time.time()
        return self
    def __exit__(self, type, value, traceback):
        print('Время обработки: {:.2f} с'.format(time.time() - self.t))

## 7. Базовые параметры

In [26]:
VOCAB_SIZE = 20000
WIN_SIZE = 1000
WIN_HOP = 100

BATCH_SIZE = 64
EPOCHS = 10

## 8. Токенизация

In [27]:
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='UNK')
tokenizer.fit_on_texts(text_train)

seq_train = tokenizer.texts_to_sequences(text_train)
seq_test = tokenizer.texts_to_sequences(text_test)

## 9. Функция создания датасета

In [28]:
def create_dataset(seq_list, win_size, win_hop):
    X = []
    y = []
    for class_id, seq in enumerate(seq_list):
        for i in range(0, len(seq) - win_size, win_hop):
            X.append(seq[i:i + win_size])
            y.append(class_id)
    return np.array(X), utils.to_categorical(y, CLASS_COUNT)

In [29]:
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.LongTensor(X)
        self.y = torch.FloatTensor(y)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

with timex():
    X_train, y_train = create_dataset(seq_train, WIN_SIZE, WIN_HOP)
    X_test, y_test = create_dataset(seq_test, WIN_SIZE, WIN_HOP)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

train_dataset = TextDataset(X_train, y_train)
test_dataset = TextDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Время обработки: 2.88 с
(18417, 1000) (18417, 6)
(6968, 1000) (6968, 6)


## 10. Модель на PyTorch

In [30]:
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, win_size, num_classes):
        super(TextClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, 128)
        self.spatial_dropout = nn.Dropout2d(0.2)
        self.conv1 = nn.Conv1d(128, 128, kernel_size=5, padding=2)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(5)
        self.lstm = nn.LSTM(128, 64, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(128, 64)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = x.permute(0, 2, 1)
        x = self.spatial_dropout(x.unsqueeze(2)).squeeze(2)
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)
        x = x.permute(0, 2, 1)
        x, _ = self.lstm(x)
        x = x[:, -1, :]
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

def create_model():
    model = TextClassifier(VOCAB_SIZE, WIN_SIZE, CLASS_COUNT).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters())
    return model, criterion, optimizer

## 11. Обучение модели

In [32]:
def train_model(model, train_loader, test_loader, criterion, optimizer, epochs):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, torch.argmax(y_batch, dim=1))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                preds = torch.argmax(outputs, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(torch.argmax(y_batch, dim=1).cpu().numpy())
        acc = accuracy_score(all_labels, all_preds)
        print(f'Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}, Val Acc: {acc:.4f}')

model, criterion, optimizer = create_model()
with timex():
    train_model(model, train_loader, test_loader, criterion, optimizer, EPOCHS)

Epoch 1/10, Loss: 1.2936, Val Acc: 0.4265
Epoch 2/10, Loss: 0.5609, Val Acc: 0.6949
Epoch 3/10, Loss: 0.1235, Val Acc: 0.7133
Epoch 4/10, Loss: 0.0701, Val Acc: 0.6863
Epoch 5/10, Loss: 0.0462, Val Acc: 0.7191
Epoch 6/10, Loss: 0.0423, Val Acc: 0.7111
Epoch 7/10, Loss: 0.0161, Val Acc: 0.6963
Epoch 8/10, Loss: 0.0090, Val Acc: 0.7117
Epoch 9/10, Loss: 0.0243, Val Acc: 0.7138
Epoch 10/10, Loss: 0.0300, Val Acc: 0.6607
Время обработки: 53.36 с


## 12. Оценка модели

In [33]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        outputs = model(X_batch)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(torch.argmax(y_batch, dim=1).cpu().numpy())
acc = accuracy_score(all_labels, all_preds)
print(f'Accuracy (VOCAB_SIZE=20000, WIN_SIZE=1000): {acc:.4f}')

Accuracy (VOCAB_SIZE=20000, WIN_SIZE=1000): 0.6607


## 13. Эксперимент с размером словаря

In [34]:
vocab_results = []

for vocab in [5000, 10000, 40000]:
    print(f'\n===== VOCAB_SIZE = {vocab} =====')

    tokenizer = Tokenizer(num_words=vocab, oov_token='UNK')
    tokenizer.fit_on_texts(text_train)
    seq_train = tokenizer.texts_to_sequences(text_train)
    seq_test = tokenizer.texts_to_sequences(text_test)

    X_train, y_train = create_dataset(seq_train, WIN_SIZE, WIN_HOP)
    X_test, y_test = create_dataset(seq_test, WIN_SIZE, WIN_HOP)

    train_dataset = TextDataset(X_train, y_train)
    test_dataset = TextDataset(X_test, y_test)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    model = TextClassifier(vocab, WIN_SIZE, CLASS_COUNT).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters())

    model.train()
    for epoch in range(EPOCHS):
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, torch.argmax(y_batch, dim=1))
            loss.backward()
            optimizer.step()

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(torch.argmax(y_batch, dim=1).cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    print(f'Accuracy: {acc:.4f}')

    vocab_results.append((vocab, acc))


===== VOCAB_SIZE = 5000 =====
Accuracy: 0.7082

===== VOCAB_SIZE = 10000 =====
Accuracy: 0.6785

===== VOCAB_SIZE = 40000 =====
Accuracy: 0.7494


## 14. Эксперимент с размером окна

In [35]:
window_results = []

for win_size, win_hop in [(500, 50), (1000, 100), (2000, 200)]:
    print(f'\n===== WIN_SIZE={win_size}, WIN_HOP={win_hop} =====')

    tokenizer = Tokenizer(num_words=20000, oov_token='UNK')
    tokenizer.fit_on_texts(text_train)
    seq_train = tokenizer.texts_to_sequences(text_train)
    seq_test = tokenizer.texts_to_sequences(text_test)

    X_train, y_train = create_dataset(seq_train, win_size, win_hop)
    X_test, y_test = create_dataset(seq_test, win_size, win_hop)

    train_dataset = TextDataset(X_train, y_train)
    test_dataset = TextDataset(X_test, y_test)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    model = TextClassifier(20000, win_size, CLASS_COUNT).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters())

    model.train()
    for epoch in range(EPOCHS):
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, torch.argmax(y_batch, dim=1))
            loss.backward()
            optimizer.step()

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(torch.argmax(y_batch, dim=1).cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    print(f'Accuracy: {acc:.4f}')

    window_results.append((win_size, win_hop, acc))


===== WIN_SIZE=500, WIN_HOP=50 =====
Accuracy: 0.7153

===== WIN_SIZE=1000, WIN_HOP=100 =====
Accuracy: 0.6929

===== WIN_SIZE=2000, WIN_HOP=200 =====
Accuracy: 0.7699


## 15. Итоговая таблица результатов

In [36]:
print('\n=== ТАБЛИЦА РЕЗУЛЬТАТОВ ===')
print('\nЭксперимент 1: Влияние размера словаря (VOCAB_SIZE)')
print(f'{"VOCAB_SIZE":<12} {"Accuracy":<10}')
print('-'*22)
for v, acc in vocab_results:
    print(f'{v:<12} {acc:.4f}')

print('\nЭксперимент 2: Влияние размера окна (WIN_SIZE, WIN_HOP)')
print(f'{"WIN_SIZE":<10} {"WIN_HOP":<10} {"Accuracy":<10}')
print('-'*35)
for w, h, acc in window_results:
    print(f'{w:<10} {h:<10} {acc:.4f}')


=== ТАБЛИЦА РЕЗУЛЬТАТОВ ===

Эксперимент 1: Влияние размера словаря (VOCAB_SIZE)
VOCAB_SIZE   Accuracy  
----------------------
5000         0.7082
10000        0.6785
40000        0.7494

Эксперимент 2: Влияние размера окна (WIN_SIZE, WIN_HOP)
WIN_SIZE   WIN_HOP    Accuracy  
-----------------------------------
500        50         0.7153
1000       100        0.6929
2000       200        0.7699
